# Section 4: Laya on Your Machine with Ollaya

**What you'll learn**
- Install Ollaya and pull the open Laya decision models
- Ask for decisions from the command line, with presets
- Call the native `/api/decide` endpoint and read its routing, timings and extras
- Point the TypeSafe SDK at your own machine
- Use case: route an online store's operations inbox locally
- Use case: read Hindi, Hinglish and English feedback through the language router
- Compare Laya and hosted Jev on the same question
- Save your own questions as a preset and as a model

## Setup
- Slides 1-5
- Install on Windows: `irm https://ollaya.dev/install.ps1 | iex`, on macOS or Linux: `curl -fsSL https://ollaya.dev/install.sh | sh`
- An `ollaya` command starts the server by itself; for the Python cells, run `ollaya serve` in a terminal first if nothing is running
- `!` runs a shell command from a notebook cell

In [1]:
!ollaya --version

ollaya version is 0.13.0


In [2]:
import asyncio, time
import httpx
import pandas as pd
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, AsyncTypeSafeClient, Choice, Score, Noul

load_dotenv()
OLLAYA = "http://localhost:11435"

## Pull and Inspect the Models
- Slides 6-11, covering Laya and How It Answers too
- `laya` is a router: pulling it also pulls `laya:en` and `laya:multilingual`
- A model that is already on disk is not downloaded again

In [3]:
!ollaya pull laya

In [4]:
!ollaya list

NAME                ID             SIZE     MODIFIED
laya:latest         89aa53ce17e1   11 KB    Less than a second ago
laya:multilingual   2840506e1f97   683 MB   Less than a second ago
laya:en             c305a9276531   853 MB   Less than a second ago


- `show laya`: the router's strategy and its two routes
- `show laya:multilingual`: parameters, context length, precision, engine

In [5]:
!ollaya show laya

  Model
    architecture        laya
    format              router
    languages           en, multilingual
    digest              sha256:89aa53ce17e105dc801b899b4c1b6d4fcbee7ea2710a303e149a5129476bc2c6

  Router
    strategy            script
    default             english
    → english           laya:en
    → multilingual      laya:multilingual

  Capabilities
    choice
    score
    noul
    act

  License
    Laya by Convai Innovations (https://huggingface.co/convaiinnovations/laya)
    Licensed under the Apache License, Version 2.0.
    ...



In [6]:
!ollaya show laya:multilingual

  Model
    architecture        laya
    parameters          322M
    context length      1024
    precision           F16/F32
    format              onnx
    engine              onnxruntime
    languages           multilingual
    digest              sha256:2840506e1f978aeb696a6f84af43ecc7fca95754cb63a67bafe6132532d384eb

  Capabilities
    choice
    score
    noul
    act

  License
    Laya by Convai Innovations (https://huggingface.co/convaiinnovations/laya)
    Licensed under the Apache License, Version 2.0.
    ...



## Decisions From the Command Line
- Slides 12-13
- `--preset triage`: five built-in support questions
- The quotes hold the whole message as one state

In [7]:
!ollaya run laya \
    "Hi, I ordered a cast iron skillet last Thursday (order 20931) and the tracking" \
    "hasn't moved since Saturday. Can someone check?" \
    --preset triage

intent            other                                 ███████████░░░░░ 0.66
is_urgent         no                                    ███████████████░ 0.94
frustration       1.18 / 3  concerned but civil         ███░░░░░░░░░░░░░ 0.17
refund_requested  no                                    █████████████░░░ 0.83
churn_risk        no                                    ███████████████░ 0.92


- `--verbose`: every option's probability, the model that answered and the timings

In [8]:
!ollaya run laya \
    "Third time I'm writing. The kettle arrived with a cracked lid and nobody replies." \
    "Refund me or I'm done with this shop." \
    --preset triage --verbose

intent            refund                                ████████████████ 1.00
                    refund                            ████████████████ 1.00
                    technical_help                    ░░░░░░░░░░░░░░░░ 0.00
                    billing_question                  ░░░░░░░░░░░░░░░░ 0.00
                    information                       ░░░░░░░░░░░░░░░░ 0.00
                    cancellation                      ░░░░░░░░░░░░░░░░ 0.00
                    other                             ░░░░░░░░░░░░░░░░ 0.00
is_urgent         no                                    ███████████░░░░░ 0.72
frustration       2.03 / 3  clearly annoyed             ██████░░░░░░░░░░ 0.37
                    0                                 █░░░░░░░░░░░░░░░ 0.03
                    1                                 ███░░░░░░░░░░░░░ 0.17
                    2                                 ████████░░░░░░░░ 0.53
                    3                                 ████░░░░░░░░░░░░ 0.26
refund


model:           laya:en (routed from laya:latest: English Latin text)
total duration:  23.1ms
load duration:   0.0ms
eval duration:   22.7ms
input tokens:    389


- `--format json`: the full response, ready for a script

In [9]:
!ollaya run laya "Can you send me a VAT invoice for order 20788?" \
    --preset triage --format json

{
  "model": "laya:en",
  "answers": {
    "intent": {
      "type": "choice",
      "choice": "other",
      "confidence": 0.2894,
      "probabilities": {
        "refund": 0.1489,
        "technical_help": 0.0129,
        "billing_question": 0.4042,
        "information": 0.0162,
        "cancellation": 0.0099,
        "other": 0.4078
      }
    },
    "is_urgent": {
      "type": "noul",
      "noul": 0.0975
    },
    "frustration": {
      "type": "score",
      "score": 1.2935,
      "confidence": 0.1684,
      "legend": {
        "0": "calm and neutral",
        "1": "concerned but civil",
        "2": "clearly annoyed",
        "3": "very angry or using strong language"
      },
      "probabilities": {
        "0": 0.2593,
        "1": 0.2761,
        "2": 0.3763,
        "3": 0.0883
      }
    },
    "refund_requested": {
      "type": "noul",
      "noul": 0.0511
    },
    "churn_risk": {
      "type": "noul",
      "noul": 0.0003
    }
  },
  "usage": {
    "input_token

- `ollaya ps`: what is loaded in memory, on which device, and until when

In [10]:
!ollaya ps

NAME                ID             SIZE     DEVICE   PRECISION   UNTIL
laya:en             c305a9276531   853 MB   cuda:0   F16         4 minutes from now
laya:multilingual   2840506e1f97   683 MB   cuda:0   F16         4 minutes from now


## The Native API: `/api/decide`
- Slide 14
- The body is the hosted Jev body: `model`, `state`, `questions`
- `timeout=60`: the first request can wait while a model loads

In [11]:
httpx.get(OLLAYA).text

'Ollaya is running'

In [12]:
body = {
    "model": "laya",
    "state": "I was charged twice for order 20966. "
             "Please refund the second payment today.",
    "questions": {
        "intent": {
            "type": "choice",
            "instructions": "What does the customer want?",
            "criteria": {
                "refund": "money back for a payment",
                "order_status": "where an order is",
                "invoice": "an invoice or a receipt",
            },
        },
        "urgency": {
            "type": "score",
            "instructions": "How urgent is this message?",
            "criteria": ["can wait", "this week", "today"],
        },
        "angry": {"type": "noul", "instructions": "The customer is angry."},
    },
}
r = httpx.post(OLLAYA + "/api/decide", json=body, timeout=60).json()
r

{'model': 'laya:en',
 'answers': {'intent': {'type': 'choice',
   'choice': 'refund',
   'confidence': 0.9716,
   'probabilities': {'refund': 0.981,
    'order_status': 0.0035,
    'invoice': 0.0154}},
  'urgency': {'type': 'score',
   'score': 1.9626,
   'confidence': 0.9549,
   'legend': {'0': 'can wait', '1': 'this week', '2': 'today'},
   'probabilities': {'0': 0.0073, '1': 0.0227, '2': 0.9699}},
  'angry': {'type': 'noul', 'noul': 0.6669}},
 'usage': {'input_tokens': 147, 'output_tokens': 0},
 'routing': {'router': 'laya:latest',
  'model': 'laya:en',
  'route': 'english',
  'reason': 'English Latin text'},
 'state_truncated': False,
 'done_reason': 'decide',
 'created_at': '2026-10-17T10:51:34.858242400Z',
 'total_duration': 25385800,
 'load_duration': 0,
 'eval_duration': 24792900}

- `model`: the checkpoint that answered, `routing`: which route the router took and why

In [13]:
r["model"], r["routing"]

('laya:en',
 {'router': 'laya:latest',
  'model': 'laya:en',
  'route': 'english',
  'reason': 'English Latin text'})

- Durations are nanoseconds: `total_duration` includes waiting, `eval_duration` is the model's own work
- Divide by `1e6` for milliseconds

In [14]:
r["total_duration"] / 1e6, r["load_duration"] / 1e6, r["eval_duration"] / 1e6

(25.3858, 0.0, 24.7929)

## Extra Fields in Laya's Response
- Slides 15-16
- `"extras": ["laya"]` adds a `laya` object to every answer: `confidence` and `act_probability`
- The model card says `act_probability` carries no usable signal yet (it reads about 1.0): gate on `confidence`

In [15]:
body["extras"] = ["laya"]
r = httpx.post(OLLAYA + "/api/decide", json=body, timeout=60).json()
r["answers"]["intent"]

{'type': 'choice',
 'choice': 'refund',
 'confidence': 0.9716,
 'probabilities': {'refund': 0.981, 'order_status': 0.0035, 'invoice': 0.0154},
 'laya': {'confidence': 0.9062, 'act_probability': 1.0}}

In [16]:
{q: a["laya"] for q, a in r["answers"].items()}

{'intent': {'confidence': 0.9062, 'act_probability': 1.0},
 'urgency': {'confidence': 0.8619, 'act_probability': 1.0},
 'angry': {'confidence': 0.6669, 'act_probability': 1.0}}

## Running Hosted Jev Code on Laya
- Slide 17
- Only `base_url`, `api_key` and `model` change
- Ollaya ignores the key, but the SDK refuses to start without one

In [17]:
local = TypeSafeClient(base_url=OLLAYA, api_key="local", model="laya")
jev = TypeSafeClient()

REFUND = Noul(instructions="The customer asks for money back.")
state = "I was charged twice for order 20966. Please refund the second payment today."

local.system_one(state, {"refund": REFUND})

SystemOneResponse(model='laya:en', usage=Usage(input_tokens=49, output_tokens=0), answers={'refund': NoulAnswer(type='noul', noul=0.8915)})

In [18]:
jev.system_one(state, {"refund": REFUND})

SystemOneResponse(model='typesafe/jev-1.14-20261014', usage=Usage(input_tokens=293, output_tokens=20), answers={'refund': NoulAnswer(type='noul', noul=0.98)})

- `/v1/models` on Ollaya lists the models on your machine

In [19]:
local.models.list()

ListModelsResponse(models=(ModelMetadata(name='laya:en', description='English decision model (ModernBERT-large): guardrails, email and ticket triage.', release_date='2026-09-23'), ModelMetadata(name='laya:latest', description="Routes each request to laya:en or laya:multilingual by the text's script and language.", release_date='2026-10-17'), ModelMetadata(name='laya:multilingual', description='Decision model for 100+ languages (mmBERT-base).', release_date='2026-09-23')))

## Use Case 1: Routing an Operations Inbox
- Slide 18
- 40 emails to a mid-size online home goods store
- `intent` is the gold label a person gave each email

In [20]:
emails = pd.read_csv("data/emails.csv")
emails.head()

,id,subject,body,intent
0,1,Where is my order #HH-20931?,"Hi, I placed an order last Thursday for the ca...",order_status
1,2,Return - wrong size,"Hi, the duvet cover I received is a double and...",return_request
2,3,VAT invoice for order 20788,"Good morning, we bought 12 bar stools for our ...",invoice_request
3,4,Collab idea: kitchen content,Hi! I run a home cooking channel on YouTube (8...,partnership
4,5,Application: Customer Service Advisor,"Dear Hiring Manager, please find attached my C...",job_application


In [21]:
emails.intent.value_counts()

intent
order_status       5
return_request     5
invoice_request    5
partnership        5
job_application    5
spam               5
complaint          5
supplier_update    5
Name: count, dtype: int64

- One description per option: say what belongs in it

In [22]:
INTENT = Choice(
    instructions="What does the sender of this email want from the store?",
    criteria={
        "order_status": "a customer asks where an order is or when it will arrive",
        "return_request": "a customer wants to send an item back, exchange it "
                          "or get a return label",
        "invoice_request": "someone needs an invoice, a receipt or a corrected invoice",
        "partnership": "a business, creator or event proposes working together "
                       "or buying wholesale",
        "job_application": "someone applies for a job or an internship",
        "spam": "unsolicited marketing, scams and phishing",
        "complaint": "a customer is unhappy about service, quality, "
                     "a charge or a person",
        "supplier_update": "a supplier announces prices, delays, "
                           "new products or changes",
    },
)

- The state is JSON: the model reads the subject and the body together

In [23]:
email = {"subject": emails.subject[0], "body": emails.body[0]}
local.system_one(email, {"intent": INTENT}).answers["intent"]

ChoiceAnswer(type='choice', choice='order_status', confidence=0.9934, probabilities={'order_status': 0.9943, 'return_request': 0.0008, 'invoice_request': 0.0005, 'partnership': 0.0003, 'job_application': 0.0003, 'spam': 0.0003, 'complaint': 0.0005, 'supplier_update': 0.0029})

### All 40 Emails
- `AsyncTypeSafeClient` takes the same three arguments
- `await` works at the top level of a notebook cell

In [24]:
alocal = AsyncTypeSafeClient(base_url=OLLAYA, api_key="local", model="laya")

results = await asyncio.gather(*[
    alocal.system_one({"subject": s, "body": b}, {"intent": INTENT})
    for s, b in zip(emails.subject, emails.body)
])
emails["pred"] = [r.answers["intent"].choice for r in results]
emails["confidence"] = [r.answers["intent"].confidence for r in results]
(emails.pred == emails.intent).mean()

np.float64(0.875)

In [25]:
pd.crosstab(emails.intent, emails.pred)

pred,complaint,invoice_request,job_application,order_status,partnership,return_request,spam,supplier_update
intent,,,,,,,,
complaint,4,0,0,0,0,0,1,0
invoice_request,0,5,0,0,0,0,0,0
job_application,0,0,5,0,0,0,0,0
order_status,0,0,0,5,0,0,0,0
partnership,0,0,0,0,4,0,1,0
return_request,0,0,0,0,0,5,0,0
spam,0,1,0,0,0,0,4,0
supplier_update,0,0,0,2,0,0,0,3


- The misses, with the confidence of each answer

In [26]:
emails[emails.pred != emails.intent][["subject", "intent", "pred", "confidence"]]

,subject,intent,pred,confidence
15,"Delay on PO 4471, linen napkins",supplier_update,order_status,0.5382
23,New AW catalogue,supplier_update,order_status,0.4508
27,Listing your products on Greenshelf,partnership,spam,0.8856
29,"Invoice #INV-88213 overdue, action required",spam,invoice_request,0.9947
38,Promo code didn't work and nobody helps,complaint,spam,0.5355


In [27]:
emails.groupby(emails.pred == emails.intent).confidence.mean()

False    0.680960
True     0.912631
Name: confidence, dtype: float64

## Use Case 2: Multilingual Customer Feedback
- Slides 19-22
- 42 reviews of an Indian food delivery app: English, Hindi, Hinglish, Tamil, Bengali
- `english` is a translation for you, the model never sees it

In [28]:
fb = pd.read_csv("data/feedback_multilingual.csv")
fb.head(6)

,id,language,text,english,topic,sentiment
0,1,english,"Order came 50 mins late, the app kept saying '...","Order came 50 mins late, the app kept saying '...",delivery_time,negative
1,2,hindi,"ऑर्डर डेढ़ घंटे बाद पहुँचा, ऐप पर बार-बार टाइम...","The order arrived after an hour and a half, an...",delivery_time,negative
2,3,hinglish,"Bhai delivery itni late kyu? 1 ghanta ho gaya,...","Brother, why is the delivery so late? It's bee...",delivery_time,negative
3,4,english,"Biryani was hot and the portion was huge, easi...","Biryani was hot and the portion was huge, easi...",food_quality,positive
4,5,hindi,"दाल मखनी बहुत स्वादिष्ट थी, बिल्कुल घर जैसा खाना।","The dal makhani was very tasty, just like home...",food_quality,positive
5,6,hinglish,"Paneer butter masala ekdum mast tha, quantity ...","The paneer butter masala was excellent, and th...",food_quality,positive


In [29]:
fb.language.value_counts()

language
hindi       14
english     13
hinglish    11
tamil        2
bengali      2
Name: count, dtype: int64

- `model_dump()` turns an SDK question into the plain JSON that `/api/decide` takes

In [30]:
TOPIC = Choice(
    instructions="What is this customer feedback about?",
    criteria={
        "delivery_time": "the order was late, early or on time",
        "food_quality": "taste, temperature, freshness or portion of the food",
        "refund_payment": "refunds, double charges, failed payments",
        "app_experience": "the app itself: crashes, speed, coupons, login, new features",
        "delivery_partner": "the rider's behaviour or help",
    },
)
SENTIMENT = Choice(
    instructions="How does the customer feel?",
    criteria={"positive": None, "neutral": None, "negative": None},
)
FEEDBACK = {"topic": TOPIC.model_dump(), "sentiment": SENTIMENT.model_dump()}
FEEDBACK["sentiment"]

{'type': 'choice',
 'instructions': 'How does the customer feel?',
 'criteria': {'positive': None, 'neutral': None, 'negative': None}}

- One Hindi review: read `routing`

In [31]:
fb.text[1], fb.english[1]

('ऑर्डर डेढ़ घंटे बाद पहुँचा, ऐप पर बार-बार टाइम बढ़ता रहा।',
 'The order arrived after an hour and a half, and the time on the app kept going up.')

In [32]:
r = httpx.post(
    OLLAYA + "/api/decide",
    json={"model": "laya", "state": fb.text[1], "questions": FEEDBACK},
    timeout=60,
).json()
r["routing"], r["answers"]

({'router': 'laya:latest',
  'model': 'laya:multilingual',
  'route': 'multilingual',
  'reason': 'non-Latin script (devanagari, 100% of letters); the English checkpoint cannot read it'},
 {'topic': {'type': 'choice',
   'choice': 'app_experience',
   'confidence': 0.9285,
   'probabilities': {'delivery_time': 0.0559,
    'food_quality': 0.0002,
    'refund_payment': 0.0002,
    'app_experience': 0.9428,
    'delivery_partner': 0.0009}},
  'sentiment': {'type': 'choice',
   'choice': 'negative',
   'confidence': 0.9997,
   'probabilities': {'positive': 0.0, 'neutral': 0.0002, 'negative': 0.9998}}})

### Every Review Through the Router
- `httpx.AsyncClient`: the same `/api/decide` call, all 42 at once

In [33]:
async with httpx.AsyncClient(timeout=60) as http:
    responses = await asyncio.gather(*[
        http.post(OLLAYA + "/api/decide",
                  json={"model": "laya", "state": t, "questions": FEEDBACK})
        for t in fb.text
    ])
answers = [x.json() for x in responses]

fb["model"] = [a["model"] for a in answers]
fb["reason"] = [a["routing"]["reason"] for a in answers]
fb["topic_pred"] = [a["answers"]["topic"]["choice"] for a in answers]
fb["topic_confidence"] = [a["answers"]["topic"]["confidence"] for a in answers]
fb["sentiment_pred"] = [a["answers"]["sentiment"]["choice"] for a in answers]
pd.crosstab(fb.language, fb.model)

model,laya:en,laya:multilingual
language,,
bengali,0,2
english,13,0
hindi,0,14
hinglish,8,3
tamil,0,2


- Hinglish is Latin script: read why each row went where it went

In [34]:
fb[fb.language == "hinglish"][["text", "model", "reason"]]

,text,model,reason
2,"Bhai delivery itni late kyu? 1 ghanta ho gaya,...",laya:en,English Latin text
5,"Paneer butter masala ekdum mast tha, quantity ...",laya:en,"Latin script, language not identified and no n..."
8,"Refund abhi tak nahi aaya, 5 din ho gaye. Cust...",laya:multilingual,"Latin script but language looks like 'ro', not..."
11,"App baar baar logout ho jaata hai, har baar OT...",laya:multilingual,"Latin script but language looks like 'it', not..."
14,"Rider bhaiya bahut helpful the, building dhoon...",laya:en,English Latin text
17,"Khana average tha, na accha na bura. Packing t...",laya:multilingual,"Latin script but language looks like 'pt', not..."
20,"Payment fail hua lekin paise kat gaye, ab kya ...",laya:en,"Latin script, language not identified and no n..."
23,"Naya dark mode accha laga, raat ko aankhon ko ...",laya:en,"Latin script, language not identified and no n..."
26,"Order time se pehle aa gaya, full marks",laya:en,"Latin script, language not identified and no n..."
29,Biryani mein chawal kachcha tha aur chicken bi...,laya:en,"Latin script, language not identified and no n..."


In [35]:
fb["topic_ok"] = fb.topic_pred == fb.topic
fb["sentiment_ok"] = fb.sentiment_pred == fb.sentiment
fb.groupby("language")[["topic_ok", "sentiment_ok"]].mean()

,topic_ok,sentiment_ok
language,,
bengali,0.500000,1.000000
english,1.000000,0.769231
hindi,0.714286,0.857143
hinglish,0.636364,0.636364
tamil,1.000000,1.000000


- What the product team reads: topic against sentiment, all languages together

In [36]:
pd.crosstab(fb.topic_pred, fb.sentiment_pred)

sentiment_pred,negative,neutral,positive
topic_pred,,,
app_experience,6,2,1
delivery_partner,1,2,5
delivery_time,6,1,2
food_quality,3,3,4
refund_payment,4,2,0


### English-Only Model on Hindi Text
- `"model": "laya:en"` skips the router

In [37]:
hindi = fb[fb.language == "hindi"].copy()

async with httpx.AsyncClient(timeout=60) as http:
    responses = await asyncio.gather(*[
        http.post(OLLAYA + "/api/decide",
                  json={"model": "laya:en", "state": t, "questions": FEEDBACK})
        for t in hindi.text
    ])
forced = [x.json() for x in responses]

hindi["en_topic"] = [a["answers"]["topic"]["choice"] for a in forced]
hindi["en_confidence"] = [a["answers"]["topic"]["confidence"] for a in forced]
hindi[["english", "topic", "topic_pred", "topic_confidence",
       "en_topic", "en_confidence"]]

,english,topic,topic_pred,topic_confidence,en_topic,en_confidence
1,"The order arrived after an hour and a half, an...",delivery_time,app_experience,0.9285,app_experience,0.1725
4,"The dal makhani was very tasty, just like home...",food_quality,food_quality,0.9977,app_experience,0.2215
7,My money was deducted but the order was not co...,refund_payment,refund_payment,1.0000,app_experience,0.2310
10,"The app is very slow, the menu alone takes hal...",app_experience,app_experience,1.0000,app_experience,0.2203
13,The delivery person brought the food on time e...,delivery_partner,delivery_partner,0.9017,app_experience,0.1972
16,"The food tasted okay, nothing special.",food_quality,food_quality,0.9985,app_experience,0.2504
19,"Money was deducted twice for the same order, p...",refund_payment,refund_payment,0.9994,app_experience,0.2444
22,"The new update is good, tracking orders is eas...",app_experience,delivery_partner,0.5860,app_experience,0.2916
25,"The food came before time, and it was hot too.",delivery_time,food_quality,0.8329,app_experience,0.2448
28,The rotis were undercooked and the curry had f...,food_quality,food_quality,0.9586,app_experience,0.2112


In [38]:
(hindi.topic_pred == hindi.topic).mean(), (hindi.en_topic == hindi.topic).mean()

(np.float64(0.7142857142857143), np.float64(0.2857142857142857))

## Laya vs Hosted Jev on the Same Question
- Slides 23-25
- 200 real bank card messages, 10 intents, typos included

In [39]:
cards = pd.read_csv("data/card_messages.csv")
cards.head()

,id,text,intent
0,1,I'm starting to think my card is lost because ...,card_arrival
1,2,What do I need to do to change my PIN?,change_pin
2,3,What do I do if I think my card was improperly...,compromised_card
3,4,I cannot find my credit card.,lost_or_stolen_card
4,5,How can I change my Tholepin ?,change_pin


In [40]:
CARD_INTENT = Choice(
    instructions="What does the bank customer need help with?",
    criteria={
        "activate_my_card": "activating a card that has arrived",
        "card_about_to_expire": "a card that expires soon, "
                                "getting a replacement before expiry",
        "card_arrival": "a new card that has not arrived yet",
        "card_not_working": "a card that does not work at all",
        "card_payment_fee_charged": "an unexpected fee on a card payment",
        "card_payment_wrong_exchange_rate": "a wrong exchange rate "
                                            "on a card payment abroad",
        "change_pin": "changing or resetting the PIN",
        "compromised_card": "someone else may have used the card or its details",
        "declined_card_payment": "a card payment that was declined",
        "lost_or_stolen_card": "a card that is lost or stolen",
    },
)

- One request each: the wall time is what a single user waits

In [41]:
%time local.system_one(cards.text[0], {"intent": CARD_INTENT}).answers["intent"].choice

CPU times: total: 0 ns
Wall time: 23.7 ms


'lost_or_stolen_card'

In [42]:
%time jev.system_one(cards.text[0], {"intent": CARD_INTENT}).answers["intent"].choice

CPU times: total: 0 ns
Wall time: 798 ms


'card_arrival'

- All 200 at once, the same code for both

In [43]:
ajev = AsyncTypeSafeClient()

start = time.perf_counter()
laya_results = await asyncio.gather(
    *[alocal.system_one(t, {"intent": CARD_INTENT}) for t in cards.text]
)
laya_seconds = time.perf_counter() - start

start = time.perf_counter()
jev_results = await asyncio.gather(
    *[ajev.system_one(t, {"intent": CARD_INTENT}) for t in cards.text]
)
jev_seconds = time.perf_counter() - start

laya_results[0].model, jev_results[0].model

('laya:en', 'typesafe/jev-1.14-20261014')

In [44]:
cards["laya"] = [r.answers["intent"].choice for r in laya_results]
cards["jev"] = [r.answers["intent"].choice for r in jev_results]

pd.DataFrame({
    "accuracy": [(cards.laya == cards.intent).mean(),
                 (cards.jev == cards.intent).mean()],
    "seconds_for_200": [laya_seconds, jev_seconds],
}, index=["laya", "jev"])

,accuracy,seconds_for_200
laya,0.875,2.241326
jev,0.970,3.875188


- Accuracy per intent, side by side

In [45]:
cards.assign(
    laya_ok=cards.laya == cards.intent,
    jev_ok=cards.jev == cards.intent,
).groupby("intent")[["laya_ok", "jev_ok"]].mean()

,laya_ok,jev_ok
intent,,
activate_my_card,0.95,0.95
card_about_to_expire,0.80,0.85
card_arrival,0.55,1.00
card_not_working,0.80,0.95
card_payment_fee_charged,0.95,1.00
card_payment_wrong_exchange_rate,0.90,0.95
change_pin,1.00,1.00
compromised_card,0.85,1.00
declined_card_payment,0.95,1.00


In [46]:
cards[cards.laya != cards.jev][["text", "intent", "laya", "jev"]].head(10)

,text,intent,laya,jev
0,I'm starting to think my card is lost because ...,card_arrival,lost_or_stolen_card,card_arrival
8,Is there tracking info available?,card_arrival,lost_or_stolen_card,card_arrival
29,How can I get my physical card to work?,card_not_working,lost_or_stolen_card,card_not_working
30,Does the card you sent have a way to track to it?,card_arrival,lost_or_stolen_card,card_arrival
37,How do I track my card?,card_arrival,lost_or_stolen_card,card_arrival
50,My card didn't work in one of the shops.,declined_card_payment,card_not_working,declined_card_payment
58,I broke my card,card_not_working,lost_or_stolen_card,card_not_working
62,going to need a new card what are the fees and...,card_about_to_expire,card_payment_fee_charged,card_about_to_expire
63,still waiting on that card,card_arrival,lost_or_stolen_card,card_arrival
68,My card doesn't accept any transaction at all....,card_not_working,declined_card_payment,card_not_working


## Presets: Saving Question Sets
- Slide 26
- Six are built in, custom ones live in the model store

In [47]:
!ollaya preset list

NAME         KIND       QUESTIONS                                                           MODIFIED
triage       built-in   intent, is_urgent, frustration, refund_requested, churn_risk
email        built-in   category, is_spam, is_phishing, urgency, needs_reply
guard        built-in   jailbreak, prompt_injection, sensitive_data, harm_severity, topic
moderation   built-in   toxic, harassment, threat, spam, severity
router       built-in   difficulty, domain, needs_tools, is_sensitive
agent        built-in   action, on_task, risk, destructive


In [48]:
!ollaya preset show email

{
  "category": {
    "type": "choice",
    "instructions": "Which team should handle the email in `body`?",
    "criteria": {
      "billing": "invoices, payments, refunds",
      "technical": "bugs, outages, integrations",
      "sales": "pricing, demos, new purchases",
      "security": "phishing, scams, account compromise",
      "hr": "hiring, leave, payroll",
      "other": "none of the above"
    }
  },
  "is_spam": {
    "type": "noul",
    "instructions": "Is this email unsolicited spam or bulk marketing?"
  },
  "is_phishing": {
    "type": "noul",
    "instructions": "Is this email a phishing or scam attempt to steal money, credentials, or personal data?",
    "criteria": {
      "true": "phishing, scam, or fraud",
      "false": "a legitimate email"
    }
  },
  "urgency": {
    "type": "score",
    "instructions": "How urgent is the request in `body`?",
    "criteria": [
      "no time pressure",
      "needs attention soon",
      "blocking issue or hard deadline"
    ]
 

email: Email: category, spam, phishing, urgency, needs a reply


- `preset` replaces `questions` on `/api/decide`
- The `email` questions mention `body`, so send the email as JSON

In [49]:
spam = emails[emails.intent == "spam"].iloc[0]
spam.subject, spam.body

('Your mailbox is almost full!!!',
 'Dear user, your mailbox has reached 98% of its storage limit. Click here to verify your account within 24 hours or your incoming emails will be deleted: http://mail-upgrade-secure.co/verify')

In [50]:
r = httpx.post(
    OLLAYA + "/api/decide",
    json={"model": "laya", "state": {"subject": spam.subject, "body": spam.body},
          "preset": "email"},
    timeout=60,
).json()
r["answers"]

{'category': {'type': 'choice',
  'choice': 'security',
  'confidence': 0.972,
  'probabilities': {'billing': 0.0041,
   'technical': 0.0081,
   'sales': 0.0036,
   'security': 0.9766,
   'hr': 0.0015,
   'other': 0.0061}},
 'is_spam': {'type': 'noul', 'noul': 0.9962},
 'is_phishing': {'type': 'noul', 'noul': 0.8949},
 'urgency': {'type': 'score',
  'score': 1.5318,
  'confidence': 0.3383,
  'legend': {'0': 'no time pressure',
   '1': 'needs attention soon',
   '2': 'blocking issue or hard deadline'},
  'probabilities': {'0': 0.027, '1': 0.4141, '2': 0.5588}},
 'needs_reply': {'type': 'noul', 'noul': 0.198}}

### Your Own Preset
- `/api/presets/create` stores a question set under a name, for every client of this server

In [51]:
httpx.post(OLLAYA + "/api/presets/create", json={
    "name": "inbox",
    "description": "Online store inbox: what the sender wants",
    "questions": {"intent": INTENT.model_dump()},
}).status_code

200

In [52]:
!ollaya run laya \
    "Hi, the invoice for order 20733 still has our old address, can you reissue it?" \
    --preset inbox

intent    invoice_request                       ████████████████ 1.00


### Building Questions Into a Model
- A Modelfile names a base model and a questions file
- `ollaya create` saves it as a new local model: a few KB of questions, routed to the same `laya:en` and `laya:multilingual`

In [53]:
!ollaya preset show inbox > inbox.json

inbox: Online store inbox: what the sender wants

In [54]:
%%writefile Modelfile
FROM laya
QUESTIONS ./inbox.json

Writing Modelfile


In [55]:
!ollaya create store-inbox -f Modelfile

using existing layer sha256:35ff836f8a432388ccf7b4a0b34a36617102972e8db35226661cf0498f29b42a
using existing layer sha256:ec56c3e69fab09b41de6f1c80fdcc13b83ec1eef1f72d3022baaafde3d9307e9
creating new layer sha256:a2f78269219178bf154d848324442979a04f22a8343558e5bbd77c144980db95
writing manifest
success


In [56]:
!ollaya run store-inbox \
    "Hello, we'd love to stock your linen range in our Lisbon shop." \
    "Do you offer trade prices?"

intent    partnership                           ███████░░░░░░░░░ 0.41


In [57]:
!ollaya list

NAME                 ID             SIZE     MODIFIED
store-inbox:latest   82cd228052a8   12 KB    Less than a second ago
laya:latest          89aa53ce17e1   11 KB    13 seconds ago
laya:multilingual    2840506e1f97   683 MB   13 seconds ago
laya:en              c305a9276531   853 MB   13 seconds ago


- `ollaya stop laya` unloads both checkpoints and frees their memory

In [58]:
!ollaya stop laya

In [59]:
!ollaya ps

NAME   ID   SIZE   DEVICE   PRECISION   UNTIL


## Cheat Sheet

| Need | Code |
|---|---|
| Install (Windows) | `irm https://ollaya.dev/install.ps1 \| iex` |
| Install (macOS, Linux) | `curl -fsSL https://ollaya.dev/install.sh \| sh` |
| Start the server | `ollaya serve` |
| Download a model | `ollaya pull laya` |
| Models on disk / in memory | `ollaya list` / `ollaya ps` |
| Model details | `ollaya show laya:en` |
| One decision in a terminal | `ollaya run laya "text" --preset triage` |
| All probabilities and timings | `ollaya run laya "text" --preset triage --verbose` |
| JSON output | `ollaya run laya "text" --preset triage --format json` |
| Native API | `httpx.post("http://localhost:11435/api/decide", json={"model": "laya", "state": s, "questions": q})` |
| Routing and timings | `r["routing"]`, `r["total_duration"]`, `r["eval_duration"]` (nanoseconds) |
| Laya's confidence and act head | `"extras": ["laya"]` then `answer["laya"]` |
| Force one checkpoint | `"model": "laya:en"` or `"model": "laya:multilingual"` |
| TypeSafe SDK, local | `TypeSafeClient(base_url="http://localhost:11435", api_key="local", model="laya")` |
| SDK question to plain JSON | `Choice(...).model_dump()` |
| List presets | `ollaya preset list` |
| Use a preset in the API | `json={"model": "laya", "state": s, "preset": "email"}` |
| Save a preset | `httpx.post(".../api/presets/create", json={"name": n, "questions": q})` |
| Model with built-in questions | Modelfile `FROM laya` + `QUESTIONS ./q.json`, then `ollaya create name -f Modelfile` |
| Unload | `ollaya stop laya` |